# Task 2 — Yelp Polarity Sentiment Classification Bake-off
**Author:** Ritika Mukesh Neema

Three from-scratch-embedding models compared: a mean-pooled-embedding baseline, a BiLSTM, and a TextCNN (Kim 2014 style). Code cells below are the real source from `src/*.py`; output cells are the real captured console output and metrics from the completed full run on the full Yelp Polarity dataset (560,000 train / 38,000 test rows, downloaded from Hugging Face `fancyzhx/yelp_polarity`).

## Preprocessing (cleaning, stemming, vocab, train/val/test split)

In [1]:
"""
Task 2.1 - Data Preprocessing for Yelp Polarity.

Handles all common raw formats:
  - HF parquet (fancyzhx/yelp_polarity: train-00000-of-00001.parquet etc.),
    columns "label","text", label in {0,1} (0=negative,1=positive)
  - HF-streamed CSV with header "label,text", label in {0,1}
  - Classic Zhang et al. Kaggle CSV, NO header, columns are label,text, label in {1,2}
    (1=negative, 2=positive)
Format is auto-detected from the file extension (.parquet vs .csv).
All are normalized to label in {0,1} with 0=negative, 1=positive.

Produces:
  - EDA plots (class distribution, review length distribution) -> outputs/
  - cleaned_train.csv / cleaned_val.csv / cleaned_test.csv (if test provided)
  - vocab.json (word -> id, built from train split only, min_freq threshold)
  - token id sequences saved as .npy (padded/truncated to max_len)
"""
import argparse
import json
import os
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer

STEMMER = PorterStemmer()
STOPWORDS = set(ENGLISH_STOP_WORDS)
_WORD_RE = re.compile(r"[a-z']+")


def load_raw_csv(path: str) -> pd.DataFrame:
    if path.lower().endswith(".parquet"):
        df = pd.read_parquet(path)
        cols = [c.lower() for c in df.columns]
        df.columns = cols
        df = df[["label", "text"]]
    else:
        # Try with header first (HF-style: label,text)
        df = pd.read_csv(path)
        cols = [c.lower() for c in df.columns]
        if "label" in cols and "text" in cols:
            df.columns = cols
            df = df[["label", "text"]]
        else:
            # No usable header -> assume classic Zhang format: label,title,text or label,text (no header)
            df = pd.read_csv(path, header=None)
            if df.shape[1] >= 3:
                df = df.iloc[:, [0, df.shape[1] - 1]]
            df.columns = ["label", "text"]

    # normalize label encoding: {1,2} (Zhang) -> {0,1}; leave {0,1} (HF) as-is
    uniq = set(pd.unique(df["label"].dropna()))
    if uniq <= {1, 2}:
        df["label"] = df["label"].map({1: 0, 2: 1})
    df["label"] = df["label"].astype(int)
    return df


def handle_missing_and_malformed(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].astype(str).str.strip().str.len() > 0]
    df = df[df["label"].isin([0, 1])]
    df = df.drop_duplicates(subset=["text"])
    df = df.reset_index(drop=True)
    print(f"  dropped {n0 - len(df)} missing/malformed/duplicate rows ({n0} -> {len(df)})")
    return df


def clean_text(text: str, remove_stopwords=True, stem=True) -> str:
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)          # stray HTML
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # urls
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = _WORD_RE.findall(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    if stem:
        # Porter stemmer: pure algorithmic, no downloaded corpus needed
        # (chosen over WordNet lemmatization specifically for offline
        # reproducibility -- see README / results.md).
        tokens = [STEMMER.stem(t) for t in tokens]
    return " ".join(tokens)


def eda(df: pd.DataFrame, out_dir: str, tag: str):
    class_counts = df["label"].value_counts().sort_index()
    lengths_words = df["text"].astype(str).str.split().map(len)
    lengths_chars = df["text"].astype(str).str.len()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(["negative (0)", "positive (1)"], class_counts.reindex([0, 1]).fillna(0).values)
    axes[0].set_title(f"Class distribution ({tag})")
    axes[0].set_ylabel("count")

    axes[1].hist(lengths_words, bins=50)
    axes[1].set_title(f"Review length in words ({tag})")
    axes[1].set_xlabel("words per review")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"eda_{tag}.png"), dpi=150)
    plt.close()

    stats = {
        "n_rows": int(len(df)),
        "class_counts": {str(k): int(v) for k, v in class_counts.items()},
        "class_balance_ratio": float(class_counts.min() / class_counts.max()) if len(class_counts) == 2 else None,
        "length_words_mean": float(lengths_words.mean()),
        "length_words_median": float(lengths_words.median()),
        "length_words_p95": float(lengths_words.quantile(0.95)),
        "length_chars_mean": float(lengths_chars.mean()),
    }
    return stats


def build_vocab(texts, min_freq=2, max_vocab=30000):
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    most_common = [w for w, c in counter.most_common() if c >= min_freq][: max_vocab - 2]
    word2idx = {"<pad>": 0, "<unk>": 1}
    for w in most_common:
        word2idx[w] = len(word2idx)
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word


def encode_texts(texts, word2idx, max_len):
    unk = word2idx["<unk>"]
    pad = word2idx["<pad>"]
    out = np.full((len(texts), max_len), pad, dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [word2idx.get(w, unk) for w in t.split()][:max_len]
        out[i, : len(ids)] = ids
        lengths[i] = len(ids)
    return out, lengths


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_csv", required=True)
    ap.add_argument("--test_csv", required=True)
    ap.add_argument("--out_dir", default="../data_processed")
    ap.add_argument("--val_frac", type=float, default=0.1, help="carved out of train for validation")
    ap.add_argument("--min_freq", type=int, default=2)
    ap.add_argument("--max_vocab", type=int, default=30000)
    ap.add_argument("--max_len", type=int, default=200)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    print("Loading...")
    train_df = load_raw_csv(args.train_csv)
    test_df = load_raw_csv(args.test_csv)

    print("Handling missing/malformed rows...")
    train_df = handle_missing_and_malformed(train_df)
    test_df = handle_missing_and_malformed(test_df)

    print("Running EDA (pre-cleaning, on raw text)...")
    train_stats = eda(train_df, args.out_dir, "train_raw")
    test_stats = eda(test_df, args.out_dir, "test_raw")
    with open(os.path.join(args.out_dir, "eda_stats.json"), "w") as f:
        json.dump({"train": train_stats, "test": test_stats}, f, indent=2)
    print("  train:", train_stats)
    print("  test:", test_stats)

    print("Cleaning text (lowercase, punctuation strip, stopword removal, Porter stemming)...")
    train_df["clean_text"] = train_df["text"].map(clean_text)
    test_df["clean_text"] = test_df["text"].map(clean_text)
    train_df = train_df[train_df["clean_text"].str.len() > 0].reset_index(drop=True)
    test_df = test_df[test_df["clean_text"].str.len() > 0].reset_index(drop=True)

    print("Carving out validation split from train...")
    rng = np.random.default_rng(args.seed)
    idx = rng.permutation(len(train_df))
    n_val = int(len(train_df) * args.val_frac)
    val_df = train_df.iloc[idx[:n_val]].reset_index(drop=True)
    train_df = train_df.iloc[idx[n_val:]].reset_index(drop=True)

    print(f"  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

    print("Building vocab from train split only...")
    word2idx, idx2word = build_vocab(train_df["clean_text"], args.min_freq, args.max_vocab)
    print(f"  vocab size = {len(word2idx)}")
    with open(os.path.join(args.out_dir, "vocab.json"), "w") as f:
        json.dump(word2idx, f)

    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        X, lengths = encode_texts(df["clean_text"], word2idx, args.max_len)
        np.save(os.path.join(args.out_dir, f"X_{name}.npy"), X)
        np.save(os.path.join(args.out_dir, f"lengths_{name}.npy"), lengths)
        np.save(os.path.join(args.out_dir, f"y_{name}.npy"), df["label"].values)
        df[["label", "text", "clean_text"]].to_csv(
            os.path.join(args.out_dir, f"cleaned_{name}.csv"), index=False
        )

    meta = {
        "vocab_size": len(word2idx),
        "max_len": args.max_len,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
        "min_freq": args.min_freq, "seed": args.seed,
    }
    with open(os.path.join(args.out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print("Done.", meta)


if __name__ == "__main__":
    main()


## Models — mean-pooled baseline, BiLSTM, TextCNN

In [1]:
"""
Task 2.2(1) - Three architecturally distinct models, all with embeddings
learned from scratch (nn.Embedding trained jointly with the classifier;
no pretrained word2vec/GloVe/transformer embeddings anywhere).

1) Baseline        : Embedding -> mean-pool -> Linear (bag-of-embeddings
                      logistic regression). No sequence order used at all.
2) Experimental #1  : BiLSTM   -> concat(final fwd, final bwd hidden) -> Linear.
                      Adds sequential / recurrent context and direction.
3) Experimental #2  : TextCNN  -> parallel 1D conv filters (kernel sizes
                      3/4/5) -> global max-pool -> concat -> Linear.
                      Adds local n-gram feature detectors instead of
                      recurrence -- a genuinely different inductive bias
                      from both the baseline and the BiLSTM.
"""
import torch
import torch.nn as nn


class BaselineMeanEmbed(nn.Module):
    """Baseline: embeddings learned from scratch, mean-pooled, linear head."""

    def __init__(self, vocab_size, embed_dim=100, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x, lengths=None):
        emb = self.embed(x)                      # (B, T, E)
        mask = (x != 0).unsqueeze(-1).float()     # (B, T, 1)
        summed = (emb * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        mean = summed / counts
        return self.fc(self.dropout(mean)).squeeze(-1)  # (B,) logits


class BiLSTMClassifier(nn.Module):
    """Experimental #1: bidirectional LSTM over the embedded sequence."""

    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, pad_idx=0,
                 num_layers=1, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, lengths):
        emb = self.embed(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        # h_n: (num_layers*2, B, hidden_dim) -> take last layer's fwd/bwd
        h_fwd, h_bwd = h_n[-2], h_n[-1]
        h = torch.cat([h_fwd, h_bwd], dim=-1)
        return self.fc(self.dropout(h)).squeeze(-1)


class TextCNNClassifier(nn.Module):
    """Experimental #2: multi-kernel 1D CNN + global max pooling (Kim, 2014 style)."""

    def __init__(self, vocab_size, embed_dim=100, num_filters=100,
                 kernel_sizes=(3, 4, 5), pad_idx=0, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList(
            [nn.Conv1d(embed_dim, num_filters, k, padding=k // 2) for k in kernel_sizes]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, x, lengths=None):
        emb = self.embed(x).transpose(1, 2)  # (B, E, T) for Conv1d
        pooled = []
        for conv in self.convs:
            c = torch.relu(conv(emb))          # (B, F, T')
            p, _ = c.max(dim=-1)                # global max-pool
            pooled.append(p)
        feat = torch.cat(pooled, dim=-1)
        return self.fc(self.dropout(feat)).squeeze(-1)


MODEL_REGISTRY = {
    "baseline": BaselineMeanEmbed,
    "bilstm": BiLSTMClassifier,
    "textcnn": TextCNNClassifier,
}


## Metrics utilities (bootstrap CIs, calibration, McNemar test)

In [1]:
"""
Task 2 - Evaluation Metrics (report all, per model).

Implements every metric in the assignment's Task 2 metrics list:
  accuracy; precision/recall/F1 (macro, micro, weighted); confusion matrix;
  ROC-AUC; PR-AUC; MCC; Brier score; expected calibration error (ECE);
  95% bootstrap CIs for accuracy/macro-F1/MCC; paired McNemar test
  (baseline vs each experimental model); macro-F1 and error rate per slice;
  parameter count / training time / examples-per-sec / peak memory (these
  three are timed in train.py and merged in here).
"""
import numpy as np
from scipy.stats import chi2, binomtest
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, average_precision_score, matthews_corrcoef, brier_score_loss,
)


def basic_classification_metrics(y_true, y_pred, y_prob):
    out = {"accuracy": float(accuracy_score(y_true, y_pred))}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=avg, zero_division=0)
        out[f"precision_{avg}"] = float(p)
        out[f"recall_{avg}"] = float(r)
        out[f"f1_{avg}"] = float(f1)
    out["confusion_matrix"] = confusion_matrix(y_true, y_pred).tolist()
    out["roc_auc"] = float(roc_auc_score(y_true, y_prob))
    out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    out["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    out["brier_score"] = float(brier_score_loss(y_true, y_prob))
    return out


def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.sum() / len(y_prob)) * abs(acc - conf)
    return float(ece)


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        stats[b] = metric_fn(y_true[idx], y_pred[idx])
    lo = float(np.percentile(stats, 100 * alpha / 2))
    hi = float(np.percentile(stats, 100 * (1 - alpha / 2)))
    return {"point": float(metric_fn(y_true, y_pred)), "ci_lower": lo, "ci_upper": hi}


def macro_f1_metric(y_true, y_pred):
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return f1


def bootstrap_all_cis(y_true, y_pred, n_boot=1000, seed=42):
    return {
        "accuracy_95ci": bootstrap_ci(y_true, y_pred, accuracy_score, n_boot, seed=seed),
        "macro_f1_95ci": bootstrap_ci(y_true, y_pred, macro_f1_metric, n_boot, seed=seed),
        "mcc_95ci": bootstrap_ci(y_true, y_pred, matthews_corrcoef, n_boot, seed=seed),
    }


def mcnemar_test(y_true, pred_a, pred_b):
    """
    Paired McNemar test comparing two models' correctness on the SAME
    validation/test examples. Uses the exact binomial test when the
    discordant-pair count is small (<25), otherwise the standard
    continuity-corrected chi-square approximation -- avoids depending on
    statsmodels, which isn't part of this environment's manifest.
    """
    y_true = np.asarray(y_true)
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(np.sum(correct_a & ~correct_b))   # a right, b wrong
    c = int(np.sum(~correct_a & correct_b))   # a wrong, b right
    n = b + c
    if n == 0:
        return {"b": b, "c": c, "statistic": 0.0, "p_value": 1.0, "method": "degenerate_no_discordant_pairs"}
    if n < 25:
        res = binomtest(min(b, c), n, 0.5)
        return {"b": b, "c": c, "statistic": None, "p_value": float(res.pvalue), "method": "exact_binomial"}
    stat = (abs(b - c) - 1) ** 2 / n
    p = float(1 - chi2.cdf(stat, df=1))
    return {"b": b, "c": c, "statistic": float(stat), "p_value": p, "method": "chi2_continuity_corrected"}


def slice_metrics(y_true, y_pred, slice_labels):
    """
    Per-slice macro-F1 and error rate. `slice_labels` is an array-like of the
    same length as y_true/y_pred giving each example's slice name (e.g. a
    review-length bucket, defined by the caller).
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    slice_labels = np.asarray(slice_labels)
    out = {}
    for s in sorted(set(slice_labels)):
        mask = slice_labels == s
        if mask.sum() == 0:
            continue
        _, _, f1, _ = precision_recall_fscore_support(
            y_true[mask], y_pred[mask], average="macro", zero_division=0
        )
        err_rate = float((y_true[mask] != y_pred[mask]).mean())
        out[str(s)] = {"n": int(mask.sum()), "macro_f1": float(f1), "error_rate": err_rate}
    return out


def length_slices(lengths, edges=(0, 20, 60, 10_000), names=("short", "medium", "long")):
    lengths = np.asarray(lengths)
    labels = np.array(["unassigned"] * len(lengths), dtype=object)
    for i, name in enumerate(names):
        mask = (lengths >= edges[i]) & (lengths < edges[i + 1])
        labels[mask] = name
    return labels


def full_report(y_true, y_pred, y_prob, lengths, n_boot=1000):
    report = basic_classification_metrics(y_true, y_pred, y_prob)
    report["ece"] = expected_calibration_error(y_true, y_prob)
    report["bootstrap_cis"] = bootstrap_all_cis(y_true, y_pred, n_boot=n_boot)
    slices = length_slices(lengths)
    report["slice_metrics_by_length"] = slice_metrics(y_true, y_pred, slices)
    return report


## Training loop (all 3 models)

In [1]:
"""
Task 2.2 - Model Training and Evaluation.

Trains all 3 models (baseline, bilstm, textcnn) on the preprocessed Yelp
Polarity split, times each one (wall-clock + examples/sec + peak memory),
evaluates each with the full Task 2 metrics list, runs the paired McNemar
test of baseline vs. each experimental model, and writes one combined
metrics_report.csv plus a per-model JSON with everything (including the
raw arrays error_analysis.py needs).
"""
import argparse
import json
import os
import resource
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from models import MODEL_REGISTRY
from metrics import full_report, mcnemar_test


class YelpDataset(Dataset):
    def __init__(self, X, y, lengths):
        self.X = torch.from_numpy(X).long()
        self.y = torch.from_numpy(y).float()
        self.lengths = torch.from_numpy(lengths).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i], self.lengths[i]


def train_one_model(name, model, train_loader, val_loader, device, epochs, lr):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    n_examples_total = 0
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb, lb in train_loader:
            xb, yb, lb = xb.to(device), yb.to(device), lb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb, lb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
            n_examples_total += xb.size(0)
        print(f"  [{name}] epoch {epoch}: loss={epoch_loss/len(train_loader.dataset):.4f}")
    train_time = time.time() - t0
    examples_per_sec = n_examples_total / train_time

    peak_mem_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    n_params = sum(p.numel() for p in model.parameters())

    return {
        "training_time_sec": train_time,
        "examples_per_sec": examples_per_sec,
        "peak_memory_mb": peak_mem_mb,
        "parameter_count": n_params,
    }


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs, preds, ys, lens = [], [], [], []
    for xb, yb, lb in loader:
        xb, lb = xb.to(device), lb.to(device)
        logits = model(xb, lb)
        p = torch.sigmoid(logits).cpu().numpy()
        probs.append(p)
        preds.append((p >= 0.5).astype(int))
        ys.append(yb.numpy())
        lens.append(lb.cpu().numpy())
    return (np.concatenate(probs), np.concatenate(preds),
            np.concatenate(ys).astype(int), np.concatenate(lens))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="../data_processed")
    ap.add_argument("--out_dir", default="../outputs")
    ap.add_argument("--ckpt_dir", default="../checkpoints")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--embed_dim", type=int, default=100)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--n_boot", type=int, default=1000)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    os.makedirs(args.out_dir, exist_ok=True)
    os.makedirs(args.ckpt_dir, exist_ok=True)

    with open(os.path.join(args.data_dir, "meta.json")) as f:
        meta = json.load(f)
    vocab_size = meta["vocab_size"]

    def load_split(name):
        X = np.load(os.path.join(args.data_dir, f"X_{name}.npy"))
        y = np.load(os.path.join(args.data_dir, f"y_{name}.npy"))
        lengths = np.load(os.path.join(args.data_dir, f"lengths_{name}.npy"))
        return X, y, lengths

    Xtr, ytr, ltr = load_split("train")
    Xva, yva, lva = load_split("val")
    Xte, yte, lte = load_split("test")

    train_loader = DataLoader(YelpDataset(Xtr, ytr, ltr), batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(YelpDataset(Xva, yva, lva), batch_size=256, shuffle=False)
    test_loader = DataLoader(YelpDataset(Xte, yte, lte), batch_size=256, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model_configs = {
        "baseline": dict(embed_dim=args.embed_dim),
        "bilstm": dict(embed_dim=args.embed_dim, hidden_dim=128),
        "textcnn": dict(embed_dim=args.embed_dim, num_filters=100, kernel_sizes=(3, 4, 5)),
    }

    all_reports = {}
    test_preds = {}   # for McNemar comparisons
    combined_rows = []

    for name, kwargs in model_configs.items():
        print(f"\n=== Training {name} ===")
        model = MODEL_REGISTRY[name](vocab_size=vocab_size, **kwargs)
        timing = train_one_model(name, model, train_loader, val_loader, device, args.epochs, args.lr)
        torch.save({"model_state": model.state_dict(), "config": kwargs, "meta": meta},
                   os.path.join(args.ckpt_dir, f"{name}.pt"))

        val_prob, val_pred, val_y, val_len = predict(model, val_loader, device)
        test_prob, test_pred, test_y, test_len = predict(model, test_loader, device)
        test_preds[name] = test_pred

        report = full_report(test_y, test_pred, test_prob, test_len, n_boot=args.n_boot)
        report.update(timing)
        report["device"] = str(device)
        all_reports[name] = report

        # keep raw arrays for error_analysis.py (not committed to the CSV report)
        np.savez(
            os.path.join(args.out_dir, f"{name}_test_predictions.npz"),
            y_true=test_y, y_pred=test_pred, y_prob=test_prob, lengths=test_len,
        )

        combined_rows.append({
            "model": name,
            "accuracy": report["accuracy"],
            "f1_macro": report["f1_macro"], "f1_micro": report["f1_micro"], "f1_weighted": report["f1_weighted"],
            "precision_macro": report["precision_macro"], "recall_macro": report["recall_macro"],
            "roc_auc": report["roc_auc"], "pr_auc": report["pr_auc"], "mcc": report["mcc"],
            "brier_score": report["brier_score"], "ece": report["ece"],
            "accuracy_ci": report["bootstrap_cis"]["accuracy_95ci"],
            "macro_f1_ci": report["bootstrap_cis"]["macro_f1_95ci"],
            "mcc_ci": report["bootstrap_cis"]["mcc_95ci"],
            "parameter_count": report["parameter_count"],
            "training_time_sec": report["training_time_sec"],
            "examples_per_sec": report["examples_per_sec"],
            "peak_memory_mb": report["peak_memory_mb"],
        })
        print(f"  test accuracy={report['accuracy']:.4f} macro-F1={report['f1_macro']:.4f} "
              f"ROC-AUC={report['roc_auc']:.4f}")

    # McNemar: baseline vs each experimental model
    mcnemar_results = {}
    for name in ["bilstm", "textcnn"]:
        mcnemar_results[f"baseline_vs_{name}"] = mcnemar_test(yte, test_preds["baseline"], test_preds[name])

    with open(os.path.join(args.out_dir, "full_metrics_report.json"), "w") as f:
        json.dump({"per_model": all_reports, "mcnemar_baseline_vs_experimental": mcnemar_results}, f, indent=2)

    pd.DataFrame(combined_rows).to_csv(os.path.join(args.out_dir, "metrics_report.csv"), index=False)

    with open(os.path.join(args.out_dir, "mcnemar_results.json"), "w") as f:
        json.dump(mcnemar_results, f, indent=2)

    print("\nDone. See outputs/metrics_report.csv, outputs/full_metrics_report.json, "
          "outputs/mcnemar_results.json, and outputs/<model>_test_predictions.npz")


if __name__ == "__main__":
    main()


## Error analysis

In [1]:
"""
Task 2.2(4) - Manual error review: 20 of your own model's errors --
5 confident false positives, 5 confident false negatives, 5 near-threshold
errors, 5 slice-specific failures. Assigns an error type to each and drafts
a testable fix (edit the drafted fix text as needed after actually reading
the reviews -- the categorization/selection logic here is exact, the fix
proposals are a starting template for your own judgement call).
"""
import argparse
import json
import os

import numpy as np
import pandas as pd

from metrics import length_slices


def select_confident_false_positives(y_true, y_pred, y_prob, k=5):
    mask = (y_pred == 1) & (y_true == 0)
    idx = np.where(mask)[0]
    idx = idx[np.argsort(-y_prob[idx])][:k]  # highest predicted-positive confidence
    return idx


def select_confident_false_negatives(y_true, y_pred, y_prob, k=5):
    mask = (y_pred == 0) & (y_true == 1)
    idx = np.where(mask)[0]
    idx = idx[np.argsort(y_prob[idx])][:k]  # lowest predicted-positive prob = most confident "negative"
    return idx


def select_near_threshold_errors(y_true, y_pred, y_prob, k=5):
    mask = y_pred != y_true
    idx = np.where(mask)[0]
    idx = idx[np.argsort(np.abs(y_prob[idx] - 0.5))][:k]
    return idx


def select_slice_specific_failures(y_true, y_pred, y_prob, lengths, k=5):
    slices = length_slices(lengths)
    err_rate_by_slice = {}
    for s in set(slices):
        m = slices == s
        if m.sum() > 0:
            err_rate_by_slice[s] = float((y_true[m] != y_pred[m]).mean())
    worst_slice = max(err_rate_by_slice, key=err_rate_by_slice.get)
    mask = (slices == worst_slice) & (y_pred != y_true)
    idx = np.where(mask)[0][:k]
    return idx, worst_slice


ERROR_TYPE_HINTS = {
    "confident_false_positive": "sarcasm / negation not captured; strong positive surface words despite a negative review",
    "confident_false_negative": "negative surface words used inside an overall positive review (e.g. 'not bad at all', comparative complaints followed by praise)",
    "near_threshold": "genuinely mixed / ambivalent review, or short review with too little signal",
    "slice_specific": "systematic weakness on this length bucket -- e.g. very short reviews give the model too few tokens to disambiguate",
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_name", default="textcnn", help="which model's predictions to review")
    ap.add_argument("--data_dir", default="../data_processed")
    ap.add_argument("--out_dir", default="../outputs")
    args = ap.parse_args()

    pred_file = os.path.join(args.out_dir, f"{args.model_name}_test_predictions.npz")
    npz = np.load(pred_file)
    y_true, y_pred, y_prob, lengths = npz["y_true"], npz["y_pred"], npz["y_prob"], npz["lengths"]

    test_df = pd.read_csv(os.path.join(args.data_dir, "cleaned_test.csv"))

    fp_idx = select_confident_false_positives(y_true, y_pred, y_prob)
    fn_idx = select_confident_false_negatives(y_true, y_pred, y_prob)
    nt_idx = select_near_threshold_errors(y_true, y_pred, y_prob)
    slice_idx, worst_slice = select_slice_specific_failures(y_true, y_pred, y_prob, lengths)

    def rows(idx_list, err_type):
        out = []
        for i in idx_list:
            out.append({
                "error_type": err_type,
                "index": int(i),
                "true_label": int(y_true[i]),
                "pred_label": int(y_pred[i]),
                "pred_prob_positive": float(y_prob[i]),
                "review_length_tokens": int(lengths[i]),
                "text": test_df.iloc[i]["text"][:500],
                "suggested_hint": ERROR_TYPE_HINTS[err_type],
            })
        return out

    all_rows = (
        rows(fp_idx, "confident_false_positive")
        + rows(fn_idx, "confident_false_negative")
        + rows(nt_idx, "near_threshold")
        + rows(slice_idx, "slice_specific")
    )

    out_path = os.path.join(args.out_dir, f"error_analysis_{args.model_name}.json")
    with open(out_path, "w") as f:
        json.dump({"worst_slice": worst_slice, "cases": all_rows}, f, indent=2)

    # also emit a markdown skeleton ready to hand-annotate with a proposed fix per case
    md_lines = [f"# Task 2.2(4) — Manual Error Review ({args.model_name})\n",
                f"Worst-performing length slice on this model: **{worst_slice}**\n"]
    for i, row in enumerate(all_rows, 1):
        md_lines.append(f"## Case {i} — {row['error_type']}")
        md_lines.append(f"- true={row['true_label']}, pred={row['pred_label']}, "
                         f"P(positive)={row['pred_prob_positive']:.3f}, length={row['review_length_tokens']} tokens")
        md_lines.append(f"- text: \"{row['text']}\"")
        md_lines.append(f"- likely cause: {row['suggested_hint']}")
        md_lines.append("- **proposed testable fix:** _(edit this after reading the review)_\n")

    md_path = os.path.join(os.path.dirname(args.out_dir) or ".", "failure_analysis.md")
    with open(md_path, "w") as f:
        f.write("\n".join(md_lines))

    print(f"Wrote {out_path} and {md_path}")
    print(f"Worst slice: {worst_slice}")


if __name__ == "__main__":
    main()


## Run: `python run_all.py --train_csv ... --test_csv ...` (full run)
Real captured console output from the completed run:

In [1]:
# Captured console output from the completed full run — not re-executed here
print(open('console_run_all_full.log').read())

$ /usr/bin/python3 preprocess.py --train_csv /content/train.parquet --test_csv /content/test.parquet

Loading...
Handling missing/malformed rows...
  dropped 0 missing/malformed/duplicate rows (560000 -> 560000)
  dropped 0 missing/malformed/duplicate rows (38000 -> 38000)
Running EDA (pre-cleaning, on raw text)...
  train: {'n_rows': 560000, 'class_counts': {'0': 280000, '1': 280000}, 'class_balance_ratio': 1.0, 'length_words_mean': 133.0288732142857, 'length_words_median': 97.0, 'length_words_p95': 372.0, 'length_chars_mean': 726.4979232142857}
  test: {'n_rows': 38000, 'class_counts': {'0': 19000, '1': 19000}, 'class_balance_ratio': 1.0, 'length_words_mean': 132.55863157894737, 'length_words_median': 97.0, 'length_words_p95': 367.0, 'length_chars_mean': 723.8446578947369}
Cleaning text (lowercase, punctuation strip, stopword removal, Porter stemming)...
Carving out validation split from train...
  train=503945  val=55993  test=37997
Building vocab from train split only...
  vocab si

### outputs/metrics_report.csv (all 3 models, full metric suite)

In [1]:
print(open('../outputs/metrics_report.csv').read())

model,accuracy,f1_macro,f1_micro,f1_weighted,precision_macro,recall_macro,roc_auc,pr_auc,mcc,brier_score,ece,accuracy_ci,macro_f1_ci,mcc_ci,parameter_count,training_time_sec,examples_per_sec,peak_memory_mb
baseline,0.9216780272126747,0.9216777537467529,0.9216780272126747,0.9216777653016509,0.9216835258976155,0.9216778854278931,0.9741014827826,0.9726264422484219,0.8433614113066465,0.058648484565828315,0.006691655138654099,"{'point': 0.9216780272126747, 'ci_lower': 0.9189396531305103, 'ci_upper': 0.9245466747374793}","{'point': 0.9216777537467529, 'ci_lower': 0.9189387515381916, 'ci_upper': 0.9245465556443613}","{'point': 0.8433614113066465, 'ci_lower': 0.8378785683546194, 'ci_upper': 0.8491010843892918}",3000101,124.85448026657104,20181.294212432356,2123.09765625
bilstm,0.933731610390294,0.9337309604013317,0.933731610390294,0.9337309440150554,0.9337493099256157,0.9337318634798293,0.981786330528643,0.9821408858073455,0.8674811732300068,0.05129483289782718,0.02529538951105921,"{'point': 0

### Paired McNemar test (baseline vs. each experimental model)

In [1]:
import json
print(json.dumps(json.load(open('../outputs/mcnemar_results.json')), indent=2))

{
  "baseline_vs_bilstm": {
    "b": 817,
    "c": 1275,
    "statistic": 99.83221797323135,
    "p_value": 0.0,
    "method": "chi2_continuity_corrected"
  },
  "baseline_vs_textcnn": {
    "b": 903,
    "c": 1068,
    "statistic": 13.645865043125317,
    "p_value": 0.00022072687364604882,
    "method": "chi2_continuity_corrected"
  }
}

### Error analysis (TextCNN) — worst slice + flagged cases
See `failure_analysis.md` for the hand-annotated write-up.

In [1]:
import json
d = json.load(open('../outputs/error_analysis_textcnn.json'))
print('worst_slice:', d['worst_slice'])
print('n_cases:', len(d['cases']))

worst_slice: long
n_cases: 20